# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [8]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [9]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [10]:
import random
import librosa

DATA_ROOT = "../data/raw/messy_mashup/genres_stems"

GENRES = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

STEMS = {
    "drums.wav": "drums",
    "vocals.wav": "vocals",
    "bass.wav": "bass",
    "other.wav": "other"
}


In [11]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    train_dataset = {g: {v: [] for v in STEMS.values()} for g in GENRES}
    val_dataset   = {g: {v: [] for v in STEMS.values()} for g in GENRES}

    rng = random.Random(seed)

    corrupted_count = 0
    less_50491_count = 0
    greater_50493_count = 0

    for genre in GENRES:

        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path):
            continue

        songs = sorted(os.listdir(genre_path))
        valid_songs = []

        for song in songs:

            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            complete = True
            corrupted_song = False

            for stem_file in STEMS.keys():

                stem_path = os.path.join(song_path, stem_file)

                if not os.path.exists(stem_path):
                    complete = False
                    continue   # do NOT break — still count others

                size_bytes = os.path.getsize(stem_path)
                size_mb = size_bytes / (1024 * 1024)

                # --------- Q1 Counting (PER STEM FILE) ---------

                # Corrupted file
                if size_bytes < 4 * 1024:
                    corrupted_count += 1
                    corrupted_song = True

                # Size conditions
                if size_mb < 5.0491:
                    less_50491_count += 1

                if size_mb > 5.0493:
                    greater_50493_count += 1

            # Only valid songs go to train/val
            if complete and not corrupted_song:
                valid_songs.append((genre, song_path))

        # --------- Train/Val Split ---------
        rng.shuffle(valid_songs)
        split_index = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_index]
        val_songs   = valid_songs[split_index:]

        def add_to_dict(target_dict, song_list):
            for genre_name, song_path in song_list:
                for stem_file, stem_name in STEMS.items():
                    stem_path = os.path.join(song_path, stem_file)
                    target_dict[genre_name][stem_name].append(stem_path)

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    # --------- PRINT ANSWERS ---------

    print("Corrupted STEM files:", corrupted_count)
    print("STEM files < 5.0491MB:", less_50491_count)
    print("STEM files > 5.0493MB:", greater_50493_count)

    print("\nQ1:")
    print(corrupted_count + less_50491_count)

    print("\nQ2:")
    print(abs(greater_50493_count - less_50491_count))

    # --------- Q3 ---------
    train_reggae_drums = len(train_dataset["reggae"]["drums"])
    val_country_vocals = len(val_dataset["country"]["vocals"])

    print("\nQ3:")
    print(abs(train_reggae_drums - val_country_vocals))

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)


Corrupted STEM files: 0
STEM files < 5.0491MB: 1256
STEM files > 5.0493MB: 184

Q1:
1256

Q2:
1072

Q3:
66


In [12]:
def merge_datasets(tr, val):
    merged = {}

    for genre in tr:
        merged[genre] = {}
        for stem in tr[genre]:
            merged[genre][stem] = tr[genre][stem] + val[genre][stem]

    return merged

In [13]:
def build_full_raw_dataset(root_dir):
    dataset = {g: {v: [] for v in STEMS.values()} for g in GENRES}

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            for stem_file, stem_name in STEMS.items():
                stem_path = os.path.join(song_path, stem_file)

                if os.path.exists(stem_path):
                    dataset[genre][stem_name].append(stem_path)

    return dataset

raw_dataset = build_full_raw_dataset(DATA_ROOT)

In [14]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """

    records = []
    total_files = 0  # ---- COUNT TOTAL FILES ----

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                total_files += 1

                # ---- Load Audio ----
                y, sr = librosa.load(file_path, sr=sr)
                total_duration = len(y) / sr

                # ---- Find Non-Silent Intervals ----
                intervals = librosa.effects.split(
                    y,
                    top_db=top_db,
                    frame_length=N_FFT,
                    hop_length=HOP_LENGTH
                )

                silence_segments = []
                silence_type = []

                # ---- CASE A: Fully silent ----
                if len(intervals) == 0:
                    silence_segments = [total_duration]
                    silence_type = ["full"]

                else:
                    # ---- CASE B: START silence ----
                    if intervals[0][0] > 0:
                        start_silence = intervals[0][0] / sr
                        silence_segments.append(start_silence)
                        silence_type.append("start")

                    # ---- CASE C: MIDDLE silence ----
                    for i in range(len(intervals) - 1):
                        gap = (intervals[i+1][0] - intervals[i][1]) / sr
                        silence_segments.append(gap)
                        silence_type.append("middle")

                    # ---- CASE D: END silence ----
                    if intervals[-1][1] < len(y):
                        end_silence = (len(y) - intervals[-1][1]) / sr
                        silence_segments.append(end_silence)
                        silence_type.append("end")

                max_silence = max(silence_segments) if silence_segments else 0

                # ---- Store result ----
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(sorted(set(silence_type))),
                        "File_Path": file_path
                    })

    print("Total files processed:", total_files)

    df = pd.DataFrame(records)
    return df
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)



Total files processed: 3320


In [15]:
print("Total rows:", len(df_silence))

Total rows: 680


In [16]:
print(df_silence.head())
print(df_silence.columns)
print("Total rows:", len(df_silence))

   Genre   Stem  Duration  Max_Silence_Sec    Silence_Location  \
0  blues  drums     30.01            16.28  end, middle, start   
1  blues  drums     30.01            12.35       middle, start   
2  blues  drums     30.01            11.26  end, middle, start   
3  blues  drums     30.01             9.57  end, middle, start   
4  blues  drums     30.01            10.89  end, middle, start   

                                           File_Path  
0  ../data/raw/messy_mashup/genres_stems/blues/bl...  
1  ../data/raw/messy_mashup/genres_stems/blues/bl...  
2  ../data/raw/messy_mashup/genres_stems/blues/bl...  
3  ../data/raw/messy_mashup/genres_stems/blues/bl...  
4  ../data/raw/messy_mashup/genres_stems/blues/bl...  
Index(['Genre', 'Stem', 'Duration', 'Max_Silence_Sec', 'Silence_Location',
       'File_Path'],
      dtype='str')
Total rows: 680


In [17]:
# Q5
print(df_silence[df_silence["Stem"] == "vocals"].shape[0])

# Q6
print(round(
    df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean(),
    2
))

# Q7
print(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
].shape[0])

# Q8
print(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
].shape[0])

# Q9
print(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0])


304
12.59
24
0
7


In [26]:
# Select first rock song from training set
rock_genre = "rock"

# Get path of first rock drum stem
first_drum_path = tr[rock_genre]["drums"][0]

# Extract base directory of the song
rock_song_dir = os.path.dirname(first_drum_path)

stems_audio = []

# Load stems in correct order
for stem in STEMS.keys():  # ['drums', 'vocals', 'bass', 'other']
    stem_path = os.path.join(rock_song_dir, stem)
    y, sr = librosa.load(
    stem_path,
    sr=SR,
    duration=DURATION
)

    stems_audio.append(y)

# ------------------- write your code here -------------------------------

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

# Print results
print("Length of mix sample:", len(mix_raw))
print("RMS Amplitude:", round(rms_val, 4))
print("Max value after normalization:", round(np.max(np.abs(mix_norm)), 4))
print("Max before normalization:", round(max_val, 2))



Length of mix sample: 110250
RMS Amplitude: 0.1021
Max value after normalization: 1.0
Max before normalization: 0.59
